# Lexical / structural / semantic evaluation: summary table by model

Computes the selected metrics for the gold/reference data and each synthetic model output.

The final table has one row per model (`Gold` plus each synthetic model) and metric summary columns containing average, minimum, maximum, and median values.

Includes `lexicalrichness` metrics: `msttr`, `mattr`, `mtld`, `hdd`, `vocd`, and `ttr`.


In [7]:
# =========================
# Configuration
# =========================

# Gold/reference files. These are concatenated into one gold set.
GOLD_JSON_FILES = [
    "../Evaluation/data/att_val_text_Nev.json",
    "../Evaluation/data/att_val_text_CF.json",
]
GOLD_TEXT_COL = "content"
GOLD_NESTED_ATTR_COL = "att_val"   

# |    1 | **Mx_Filter**   |    **96.68** |             **998** |          674 |
# |    2 | *MG_RAG_Filter* |      *96.66* |               *993* |      **689** |
# |    3 | Mx_RAG_Filter   |        96.64 |                 990 |        *688* |
# |    4 | MG_Filter       |        95.97 |                 973 |          605 |
# |    5 | MG              |        94.63 |                 915 |          485 |
# |    6 | MG_RAG          |        94.44 |                 900 |          490 |

# Synthetic files to compare with gold.
SYNTHETIC_SETS = {
    "MedGemma": "LLMEvaluation/noSel_noRag_medgemma.csv",
    "MedGemma_Filt": "LLMEvaluation/selectedMedGemma.csv",
    "MedGemma_RAG": "LLMEvaluation/no_sel_RAG_medgemma.csv",
    "MedGemma_RAG_Filt": "LLMEvaluation/selectedMedGemma_RAG.csv",
    "Mixtral": "LLMEvaluation/noSel_noRag_mixtral.csv",
    "Mixtral_Filt": "LLMEvaluation/selectedMixtral.csv",
    "Mixtral_RAG": "LLMEvaluation/no_sel_RAG_mixtral.csv",
    "Mixtral_RAG_Filt": "LLMEvaluation/selectedMixtral_RAG.csv",
}

SYNTH_TEXT_COL = "text"

# Choose metrics: Use "all" to compute every available metric. Or provide a list
# lexicalrichness metrics are also included in "all": msttr, mattr, mtld, hdd, vocd, ttr.
# (install with: pip install lexicalrichness)
SELECTED_METRICS = "all"

ENABLE_PERPLEXITY = True
ENABLE_EMBEDDINGS = False

PERPLEXITY_MODEL_NAME = "bigscience/bloom-7b1"
PERPLEXITY_MAX_LENGTH = 512      
PERPLEXITY_BATCH_SIZE = 4

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 32

OUTPUT_CSV = "teacher_model_metric_summary_table.csv"


In [8]:
import math
import re
import zlib
from functools import lru_cache
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, mannwhitneyu

try:
    from lexicalrichness import LexicalRichness
except ImportError:
    LexicalRichness = None


def _flatten_nested_attribute_column(df: pd.DataFrame, nested_col: str | None) -> pd.DataFrame:
    """Expand a column containing dictionaries, if present."""
    if nested_col and nested_col in df.columns:
        expanded = df[nested_col].apply(pd.Series)
        df = df.drop(columns=[nested_col]).join(expanded)
    return df


def load_gold_texts(files, text_col: str, nested_col: str | None = None) -> pd.Series:
    frames = []
    for file in files:
        df = pd.read_json(file)
        df = _flatten_nested_attribute_column(df, nested_col)
        if text_col not in df.columns:
            raise ValueError(f"Gold text column '{text_col}' not found in {file}. Available columns: {list(df.columns)}")
        frames.append(df)
    gold_df = pd.concat(frames, ignore_index=True)
    return clean_text_series(gold_df[text_col])


def load_synthetic_texts(synthetic_sets: dict[str, str], text_col: str) -> dict[str, pd.Series]:
    out = {}
    for label, file in synthetic_sets.items():
        df = pd.read_csv(file)
        if text_col not in df.columns:
            raise ValueError(f"Synthetic text column '{text_col}' not found in {file}. Available columns: {list(df.columns)}")
        out[label] = clean_text_series(df[text_col])
    return out


def clean_text_series(values) -> pd.Series:
    """Keep only non-empty strings."""
    return (
        pd.Series(values)
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.str.len() > 0]
        .reset_index(drop=True)
    )


gold_texts = load_gold_texts(GOLD_JSON_FILES, GOLD_TEXT_COL, GOLD_NESTED_ATTR_COL)
synthetic_texts_by_set = load_synthetic_texts(SYNTHETIC_SETS, SYNTH_TEXT_COL)

print(f"Gold documents: {len(gold_texts)}")
for name, texts in synthetic_texts_by_set.items():
    print(f"{name}: {len(texts)} documents")

Gold documents: 377
MedGemma: 1062 documents
MedGemma_Filt: 690 documents
MedGemma_RAG: 1062 documents
MedGemma_RAG_Filt: 662 documents
Mixtral: 1062 documents
Mixtral_Filt: 561 documents
Mixtral_RAG: 1062 documents
Mixtral_RAG_Filt: 408 documents


In [9]:

# =========================
#    Metric definitions
# =========================

token_pattern = re.compile(r"\b\w+\b")
sentence_splitter = re.compile(r"(?<=[.!?])\s+|\n+")


def tokenize(text: str) -> list[str]:
    return token_pattern.findall(str(text).lower())


def split_sentences(text: str) -> list[str]:
    parts = sentence_splitter.split(str(text).strip())
    return [p.strip() for p in parts if p.strip()]


def ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


def safe_mean(values):
    return float(np.mean(values)) if values else np.nan


def safe_median(values):
    return float(np.median(values)) if values else np.nan


def safe_std(values):
    return float(np.std(values)) if values else np.nan


def repetition_ratio(tokens: list[str], n: int) -> float:
    grams = ngrams(tokens, n)
    if not grams:
        return np.nan
    return 1 - (len(set(grams)) / len(grams))


def shannon_entropy(tokens: list[str]) -> float:
    if not tokens:
        return np.nan
    counts = np.array(list(Counter(tokens).values()), dtype=float)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log(probs)))


def ngram_diversity_score(tokens: list[str], max_n: int = 4) -> float:
    scores = []
    for n in range(1, max_n + 1):
        grams = ngrams(tokens, n)
        if grams:
            scores.append(len(set(grams)) / len(grams))
    return float(np.mean(scores)) if scores else np.nan


def compression_ratio(text: str) -> float:
    if not text:
        return np.nan
    raw = str(text).encode("utf-8")
    if len(raw) == 0:
        return np.nan
    return len(zlib.compress(raw)) / len(raw)


def pattr(tokens: list[str], target_length: int = 100) -> float:
    if not tokens:
        return np.nan
    return len(set(tokens)) / (len(tokens) + abs(len(tokens) - target_length))


# LEXICAL_RICHNESS_METRICS = ["msttr", "mattr", "mtld", "hdd", "vocd", "ttr"]
LEXICAL_RICHNESS_METRICS = ["mtld"]


def lexical_metrics(text):
    """Compute lexicalrichness metrics for one document.

    Returns NaN for all metrics if the text is empty, lexicalrichness is not installed,
    or lexicalrichness cannot compute a metric for the document.
    """
    row = {
        "msttr": np.nan,
        "mattr": np.nan,
        "mtld": np.nan,
        "hdd": np.nan,
        "vocd": np.nan,
        "ttr": np.nan,
    }

    if not isinstance(text, str) or not text.strip() or LexicalRichness is None:
        return row

    try:
        lex = LexicalRichness(text)
        row["msttr"] = lex.msttr()
        row["mattr"] = lex.mattr()
        row["mtld"] = lex.mtld()
        row["hdd"] = lex.hdd()
        row["vocd"] = lex.vocd()
        row["ttr"] = lex.ttr
    except Exception:
        pass

    return row


def document_metrics(text: str) -> dict[str, float]:
    tokens = tokenize(text)
    sentences = split_sentences(text)
    sentence_lengths = [len(tokenize(sentence)) for sentence in sentences if tokenize(sentence)]
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    line_lengths = [len(tokenize(line)) for line in lines if tokenize(line)]
    numeric_tokens = re.findall(r"\b\d+(?:\.\d+)?\b", str(text))
    token_count = len(tokens)
    unique_count = len(set(tokens))

    metrics = {
        # Lexical diversity and repetition
        "length": token_count,
        "unique_tokens": unique_count,
        # "ttr": unique_count / token_count if token_count else np.nan,
        # "entropy": shannon_entropy(tokens),
        # "hill_1": math.exp(shannon_entropy(tokens)) if token_count else np.nan,
        # "hill_2": 1 / np.sum((np.array(list(Counter(tokens).values()), dtype=float) / token_count) ** 2) if token_count else np.nan,
        # "rep_bigram": repetition_ratio(tokens, n=2),
        # "rep_trigram": repetition_ratio(tokens, n=3),
        # "repetition": 1 - (unique_count / token_count) if token_count else np.nan,
        "ngram_diversity": ngram_diversity_score(tokens),
        "compression_ratio": compression_ratio(text),
        # "pattr": pattr(tokens),

        # Structural variation
        "n_sentences": len(sentences),
        "mean_sent_len": safe_mean(sentence_lengths),
        # "median_sent_len": safe_median(sentence_lengths),
        # "std_sent_len": safe_std(sentence_lengths),
        # "max_sent_len": float(np.max(sentence_lengths)) if sentence_lengths else np.nan,
        # "n_lines": len(lines),
        # "mean_line_len": safe_mean(line_lengths),
        # "colon_count": str(text).count(":"),
        # "semicolon_count": str(text).count(";"),
        # "paren_count": str(text).count("(") + str(text).count(")"),
        # "numeric_ratio": len(numeric_tokens) / token_count if token_count else np.nan,
    }

    # Overwrite `ttr` with LexicalRichness' implementation and add msttr/mattr/mtld/hdd/vocd.
    metrics.update(lexical_metrics(text))
    return metrics


BASE_METRICS = list(document_metrics("Example sentence.").keys())
PERPLEXITY_METRICS = ["perplexity"] if ENABLE_PERPLEXITY else []
EMBEDDING_METRICS = [
    "embedding_cosine_to_gold_centroid",
    "embedding_euclidean_to_gold_centroid",
    "embedding_nearest_gold_cosine",
] if ENABLE_EMBEDDINGS else []

AVAILABLE_METRICS = BASE_METRICS + PERPLEXITY_METRICS + EMBEDDING_METRICS

if SELECTED_METRICS == "all":
    METRICS_TO_USE = AVAILABLE_METRICS
else:
    missing = sorted(set(SELECTED_METRICS) - set(AVAILABLE_METRICS))
    if missing:
        raise ValueError(f"Unknown metrics: {missing}. Available metrics are: {AVAILABLE_METRICS}")
    METRICS_TO_USE = list(SELECTED_METRICS)

BASE_METRICS_TO_USE = [m for m in METRICS_TO_USE if m in BASE_METRICS]
USE_PERPLEXITY = "perplexity" in METRICS_TO_USE
USE_EMBEDDINGS = any(m in METRICS_TO_USE for m in EMBEDDING_METRICS)

print("Selected metrics:")
print(METRICS_TO_USE)


Selected metrics:
['length', 'unique_tokens', 'ngram_diversity', 'compression_ratio', 'n_sentences', 'mean_sent_len', 'msttr', 'mattr', 'mtld', 'hdd', 'vocd', 'ttr', 'perplexity']


In [10]:
# Build one summary table

def base_metrics_dataframe(texts: pd.Series, metrics_to_use: list[str]) -> pd.DataFrame:
    if not metrics_to_use:
        return pd.DataFrame(index=range(len(texts)))
    rows = [document_metrics(text) for text in texts]
    return pd.DataFrame(rows)[metrics_to_use]


def compute_perplexity_series(texts: pd.Series) -> pd.Series:
    """Compute per-document perplexity with a Hugging Face causal language model."""
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as exc:
        raise ImportError(
            "Perplexity requires transformers and torch. Install them or remove 'perplexity' from SELECTED_METRICS."
        ) from exc

    # device = "cuda" if torch.cuda.is_available() else "cpu"
    device = "cuda:0"
    tokenizer = AutoTokenizer.from_pretrained(PERPLEXITY_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(PERPLEXITY_MODEL_NAME).to(device)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    scores = []
    for start in range(0, len(texts), PERPLEXITY_BATCH_SIZE):
        batch = list(texts.iloc[start : start + PERPLEXITY_BATCH_SIZE].astype(str))
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=PERPLEXITY_MAX_LENGTH,
        ).to(device)

        labels = encoded["input_ids"].clone()
        labels[encoded["attention_mask"] == 0] = -100

        with torch.no_grad():
            logits = model(**encoded).logits[:, :-1, :]
            shifted_labels = labels[:, 1:]
            loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
            token_losses = loss_fct(logits.reshape(-1, logits.size(-1)), shifted_labels.reshape(-1)).reshape(shifted_labels.shape)
            valid_tokens = shifted_labels.ne(-100)
            doc_losses = (token_losses * valid_tokens).sum(dim=1) / valid_tokens.sum(dim=1).clamp(min=1)
            batch_ppl = torch.exp(doc_losses).detach().cpu().numpy()
        scores.extend(batch_ppl.tolist())

    return pd.Series(scores, name="perplexity")

def compute_embedding_metric_frames(gold_texts: pd.Series, synthetic_texts_by_set: dict[str, pd.Series]) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Compute embedding similarity/distance metrics against the gold embedding distribution."""
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "Embedding comparison requires sentence-transformers. Install it or remove embedding metrics from SELECTED_METRICS."
        ) from exc

    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    def encode(texts: pd.Series) -> np.ndarray:
        return model.encode(
            list(texts.astype(str)),
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    gold_emb = encode(gold_texts)
    gold_centroid = gold_emb.mean(axis=0)
    gold_centroid = gold_centroid / np.linalg.norm(gold_centroid)

    # Gold baseline: each gold document compared to the gold centroid and, for nearest-neighbor,
    # to the nearest *other* gold document.
    gold_cosine_centroid = gold_emb @ gold_centroid
    gold_euclidean_centroid = np.linalg.norm(gold_emb - gold_centroid, axis=1)
    gold_similarity_matrix = gold_emb @ gold_emb.T
    np.fill_diagonal(gold_similarity_matrix, np.nan)
    gold_nearest = np.nanmax(gold_similarity_matrix, axis=1)

    gold_frame = pd.DataFrame({
        "embedding_cosine_to_gold_centroid": gold_cosine_centroid,
        "embedding_euclidean_to_gold_centroid": gold_euclidean_centroid,
        "embedding_nearest_gold_cosine": gold_nearest,
    })

    synth_frames = {}
    for synth_name, synth_texts in synthetic_texts_by_set.items():
        synth_emb = encode(synth_texts)
        synth_frames[synth_name] = pd.DataFrame({
            "embedding_cosine_to_gold_centroid": synth_emb @ gold_centroid,
            "embedding_euclidean_to_gold_centroid": np.linalg.norm(synth_emb - gold_centroid, axis=1),
            "embedding_nearest_gold_cosine": np.max(synth_emb @ gold_emb.T, axis=1),
        })

    return gold_frame, synth_frames


def summarize_metric_frame(metric_frame: pd.DataFrame) -> pd.Series:
    """Return avg/min/max/median for every selected metric in one row."""
    summary = {}
    for metric in METRICS_TO_USE:
        values = pd.to_numeric(metric_frame[metric], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        summary[(metric, "average")] = values.mean() if len(values) else np.nan
        summary[(metric, "min")] = values.min() if len(values) else np.nan
        summary[(metric, "max")] = values.max() if len(values) else np.nan
        summary[(metric, "median")] = values.median() if len(values) else np.nan
    return pd.Series(summary)


# 1) Fast lexical/structural metrics for gold and each synthetic model.
gold_metrics = base_metrics_dataframe(gold_texts, BASE_METRICS_TO_USE)
synth_metrics_by_set = {
    name: base_metrics_dataframe(texts, BASE_METRICS_TO_USE)
    for name, texts in synthetic_texts_by_set.items()
}

# 2) Optional perplexity.
if USE_PERPLEXITY:
    gold_metrics["perplexity"] = compute_perplexity_series(gold_texts)
    for name, texts in synthetic_texts_by_set.items():
        synth_metrics_by_set[name]["perplexity"] = compute_perplexity_series(texts)

# 3) Optional embedding comparison.
if USE_EMBEDDINGS:
    gold_embedding_metrics, synth_embedding_metrics_by_set = compute_embedding_metric_frames(gold_texts, synthetic_texts_by_set)
    for metric in EMBEDDING_METRICS:
        if metric in METRICS_TO_USE:
            gold_metrics[metric] = gold_embedding_metrics[metric]
            for name in synthetic_texts_by_set:
                synth_metrics_by_set[name][metric] = synth_embedding_metrics_by_set[name][metric]

# 4) Final table: one row for Gold, then one row per synthetic model.
metrics_by_model = {"Gold": gold_metrics, **synth_metrics_by_set}

final_table = pd.DataFrame.from_dict(
    {model: summarize_metric_frame(metric_frame) for model, metric_frame in metrics_by_model.items()},
    orient="index",
)
final_table.index.name = "model"
final_table.columns = pd.MultiIndex.from_tuples(final_table.columns, names=["metric", "statistic"])
final_table = final_table.round(2)

# Save a CSV 
final_table_for_csv = final_table.copy()
final_table_for_csv.columns = [f"{metric}_{statistic}" for metric, statistic in final_table_for_csv.columns]
final_table_for_csv.to_csv(OUTPUT_CSV)

final_table


Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

metric             length                      unique_tokens                \
statistic         average    min    max median       average    min    max   
model                                                                        
Gold               298.18   35.0  932.0  266.0        164.13   28.0  353.0   
MedGemma           221.62  152.0  298.0  224.0        130.06  100.0  159.0   
MedGemma_Filt      223.01  152.0  298.0  226.0        130.38  102.0  159.0   
MedGemma_RAG       155.50   63.0  259.0  154.0        105.48   47.0  170.0   
MedGemma_RAG_Filt  155.92   82.0  256.0  155.5        105.73   56.0  163.0   
Mixtral            217.40  130.0  351.0  219.0        109.23   71.0  156.0   
Mixtral_Filt       215.56  130.0  326.0  214.0        109.12   71.0  143.0   
Mixtral_RAG        193.57   14.0  312.0  198.0        121.07   12.0  172.0   
Mixtral_RAG_Filt   184.04   76.0  294.0  181.5        114.08   63.0  170.0   

metric                   ngram_diversity        ...    vocd             ttr  \
statistic         median         average   min  ...     max  median average   
model                                           ...                           
Gold               160.0            0.84  0.56  ...  248.05  117.98    0.59   
MedGemma           130.0            0.85  0.71  ...  278.99  118.61    0.61   
MedGemma_Filt      130.0            0.85  0.71  ...  278.99  118.74    0.60   
MedGemma_RAG       102.0            0.89  0.68  ...  232.85  125.85    0.69   
MedGemma_RAG_Filt  103.0            0.89  0.68  ...  232.85  126.17    0.69   
Mixtral            109.0            0.76  0.53  ...  161.67   78.51    0.51   
Mixtral_Filt       109.0            0.76  0.53  ...  134.64   80.76    0.51   
Mixtral_RAG        122.0            0.85  0.67  ...  202.97  118.57    0.64   
Mixtral_RAG_Filt   108.0            0.85  0.68  ...  202.97  109.72    0.63   

metric                               perplexity                      
statistic           min   max median    average   min    max median  
model                                                                
Gold               0.27  0.84   0.58      15.35  5.07  54.28  14.24  
MedGemma           0.46  0.80   0.60       4.70  3.61   6.43   4.65  
MedGemma_Filt      0.46  0.80   0.59       4.67  3.65   6.43   4.61  
MedGemma_RAG       0.48  0.82   0.70       7.86  4.05  13.30   7.63  
MedGemma_RAG_Filt  0.48  0.82   0.70       7.80  4.05  13.30   7.51  
Mixtral            0.35  0.67   0.51       3.98  2.83   6.73   3.96  
Mixtral_Filt       0.35  0.67   0.51       4.00  2.88   5.50   3.98  
Mixtral_RAG        0.44  0.82   0.65       7.58  3.34  39.41   7.42  
Mixtral_RAG_Filt   0.47  0.82   0.64       6.90  3.28  14.10   6.61  

[9 rows x 52 columns]

In [11]:
final_table

metric             length                      unique_tokens                \
statistic         average    min    max median       average    min    max   
model                                                                        
Gold               298.18   35.0  932.0  266.0        164.13   28.0  353.0   
MedGemma           221.62  152.0  298.0  224.0        130.06  100.0  159.0   
MedGemma_Filt      223.01  152.0  298.0  226.0        130.38  102.0  159.0   
MedGemma_RAG       155.50   63.0  259.0  154.0        105.48   47.0  170.0   
MedGemma_RAG_Filt  155.92   82.0  256.0  155.5        105.73   56.0  163.0   
Mixtral            217.40  130.0  351.0  219.0        109.23   71.0  156.0   
Mixtral_Filt       215.56  130.0  326.0  214.0        109.12   71.0  143.0   
Mixtral_RAG        193.57   14.0  312.0  198.0        121.07   12.0  172.0   
Mixtral_RAG_Filt   184.04   76.0  294.0  181.5        114.08   63.0  170.0   

metric                   ngram_diversity        ...    vocd             ttr  \
statistic         median         average   min  ...     max  median average   
model                                           ...                           
Gold               160.0            0.84  0.56  ...  248.05  117.98    0.59   
MedGemma           130.0            0.85  0.71  ...  278.99  118.61    0.61   
MedGemma_Filt      130.0            0.85  0.71  ...  278.99  118.74    0.60   
MedGemma_RAG       102.0            0.89  0.68  ...  232.85  125.85    0.69   
MedGemma_RAG_Filt  103.0            0.89  0.68  ...  232.85  126.17    0.69   
Mixtral            109.0            0.76  0.53  ...  161.67   78.51    0.51   
Mixtral_Filt       109.0            0.76  0.53  ...  134.64   80.76    0.51   
Mixtral_RAG        122.0            0.85  0.67  ...  202.97  118.57    0.64   
Mixtral_RAG_Filt   108.0            0.85  0.68  ...  202.97  109.72    0.63   

metric                               perplexity                      
statistic           min   max median    average   min    max median  
model                                                                
Gold               0.27  0.84   0.58      15.35  5.07  54.28  14.24  
MedGemma           0.46  0.80   0.60       4.70  3.61   6.43   4.65  
MedGemma_Filt      0.46  0.80   0.59       4.67  3.65   6.43   4.61  
MedGemma_RAG       0.48  0.82   0.70       7.86  4.05  13.30   7.63  
MedGemma_RAG_Filt  0.48  0.82   0.70       7.80  4.05  13.30   7.51  
Mixtral            0.35  0.67   0.51       3.98  2.83   6.73   3.96  
Mixtral_Filt       0.35  0.67   0.51       4.00  2.88   5.50   3.98  
Mixtral_RAG        0.44  0.82   0.65       7.58  3.34  39.41   7.42  
Mixtral_RAG_Filt   0.47  0.82   0.64       6.90  3.28  14.10   6.61  

[9 rows x 52 columns]

In [13]:
def format_final_table_metric_rows(final_table, model_col="model"):
    """
    Reformat final_table for presentation only:
    rows = metrics/statistics
    columns = models
    """
    table = final_table.copy()

    if model_col in table.columns:
        table = table.set_index(model_col)

    return table.T

In [14]:
presentation_table = format_final_table_metric_rows(final_table)
presentation_table.to_csv("final_Teachers_table_metrics_rows_models_columns.csv")
presentation_table

model                          Gold  MedGemma  MedGemma_Filt  MedGemma_RAG  \
metric            statistic                                                  
length            average    298.18    221.62         223.01        155.50   
                  min         35.00    152.00         152.00         63.00   
                  max        932.00    298.00         298.00        259.00   
                  median     266.00    224.00         226.00        154.00   
unique_tokens     average    164.13    130.06         130.38        105.48   
                  min         28.00    100.00         102.00         47.00   
                  max        353.00    159.00         159.00        170.00   
                  median     160.00    130.00         130.00        102.00   
ngram_diversity   average      0.84      0.85           0.85          0.89   
                  min          0.56      0.71           0.71          0.68   
                  max          0.97      0.93           0.93          0.96   
                  median       0.84      0.85           0.85          0.89   
compression_ratio average      0.54      0.51           0.51          0.57   
                  min          0.32      0.43           0.43          0.44   
                  max          0.85      0.57           0.57          0.66   
                  median       0.54      0.51           0.51          0.57   
n_sentences       average     40.73     28.48          28.54         22.49   
                  min          7.00     22.00          22.00         13.00   
                  max        121.00     37.00          37.00         32.00   
                  median      36.00     28.00          29.00         23.00   
mean_sent_len     average      7.47      7.78           7.81          6.85   
                  min          3.70      6.03           6.03          4.17   
                  max         11.80     10.32          10.32         10.08   
                  median       7.30      7.74           7.77          6.86   
msttr             average      0.74      0.75           0.74          0.75   
                  min          0.58      0.60           0.62          0.42   
                  max          0.89      0.89           0.89          0.88   
                  median       0.74      0.72           0.72          0.75   
mattr             average      0.73      0.71           0.71          0.75   
                  min          0.56      0.60           0.62          0.54   
                  max          0.87      0.85           0.85          0.85   
                  median       0.73      0.71           0.71          0.75   
mtld              average    101.19     86.89          86.92        109.97   
                  min         35.79     38.94          38.94         24.84   
                  max        213.16    189.00         189.00        218.41   
                  median      97.17     84.46          84.46        104.11   
hdd               average      0.86      0.87           0.87          0.87   
                  min          0.76      0.79           0.79          0.75   
                  max          0.93      0.94           0.94          0.92   
                  median       0.87      0.87           0.87          0.87   
vocd              average    120.52    122.62         122.36        126.98   
                  min         49.91     64.67          64.67         45.97   
                  max        248.05    278.99         278.99        232.85   
                  median     117.98    118.61         118.74        125.85   
ttr               average      0.59      0.61           0.60          0.69   
                  min          0.27      0.46           0.46          0.48   
                  max          0.84      0.80           0.80          0.82   
                  median       0.58      0.60           0.59          0.70   
perplexity        average     15.35      4.70           4.67          7.86   
                  min 

In [15]:
def format_final_table_metric_rows_with_subcols(
    final_table,
    model_col="model",
    stats=("average", "max", "min","median"),  #"max", "min", 
):
    """
    Presentation-only reformat:
    rows = metrics
    columns = MultiIndex: model -> avg/max/min/mean

    Handles both:
    1. flat columns like "msttr_avg"
    2. tuple/MultiIndex columns like ("msttr", "avg")
    """
    table = final_table.copy()

    if model_col in table.columns:
        table = table.set_index(model_col)

    out = {}

    for model in table.index:
        model_values = table.loc[model]

        for col in table.columns:
            # Case 1: columns are tuples, e.g. ("msttr", "avg")
            if isinstance(col, tuple):
                metric = col[0]
                stat = col[1]

                if stat in stats:
                    out.setdefault(metric, {})[(model, stat)] = model_values[col]

            # Case 2: columns are strings, e.g. "msttr_avg"
            else:
                col_str = str(col)

                for stat in stats:
                    suffix = f"_{stat}"

                    if col_str.endswith(suffix):
                        metric = col_str[: -len(suffix)]
                        out.setdefault(metric, {})[(model, stat)] = model_values[col]
                        break

    presentation_table = pd.DataFrame.from_dict(out, orient="index")

    presentation_table.columns = pd.MultiIndex.from_tuples(
        presentation_table.columns,
        names=["model", "stat"]
    )

    presentation_table = presentation_table.sort_index(axis=1, level=[0, 1])

    return presentation_table

In [16]:
presentation_table2 = format_final_table_metric_rows_with_subcols(final_table)
presentation_table2.to_csv("final_teachers_table_metrics_rows_models_columns2.csv")
presentation_table2

model                Gold                        MedGemma                  \
stat              average     max  median    min  average     max  median   
length             298.18  932.00  266.00  35.00   221.62  298.00  224.00   
unique_tokens      164.13  353.00  160.00  28.00   130.06  159.00  130.00   
ngram_diversity      0.84    0.97    0.84   0.56     0.85    0.93    0.85   
compression_ratio    0.54    0.85    0.54   0.32     0.51    0.57    0.51   
n_sentences         40.73  121.00   36.00   7.00    28.48   37.00   28.00   
mean_sent_len        7.47   11.80    7.30   3.70     7.78   10.32    7.74   
msttr                0.74    0.89    0.74   0.58     0.75    0.89    0.72   
mattr                0.73    0.87    0.73   0.56     0.71    0.85    0.71   
mtld               101.19  213.16   97.17  35.79    86.89  189.00   84.46   
hdd                  0.86    0.93    0.87   0.76     0.87    0.94    0.87   
vocd               120.52  248.05  117.98  49.91   122.62  278.99  118.61   
ttr                  0.59    0.84    0.58   0.27     0.61    0.80    0.60   
perplexity          15.35   54.28   14.24   5.07     4.70    6.43    4.65   

model                     MedGemma_Filt          ... Mixtral_Filt          \
stat                  min       average     max  ...       median     min   
length             152.00        223.01  298.00  ...       214.00  130.00   
unique_tokens      100.00        130.38  159.00  ...       109.00   71.00   
ngram_diversity      0.71          0.85    0.93  ...         0.78    0.53   
compression_ratio    0.43          0.51    0.57  ...         0.46    0.34   
n_sentences         22.00         28.54   37.00  ...        23.00   14.00   
mean_sent_len        6.03          7.81   10.32  ...         9.40    5.32   
msttr                0.60          0.74    0.89  ...         0.68    0.53   
mattr                0.60          0.71    0.85  ...         0.67    0.55   
mtld                38.94         86.92  189.00  ...        66.72   33.79   
hdd                  0.79          0.87    0.94  ...         0.82    0.75   
vocd                64.67        122.36  278.99  ...        80.76   46.58   
ttr                  0.46          0.60    0.80  ...         0.51    0.35   
perplexity           3.61          4.67    6.43  ...         3.98    2.88   

model             Mixtral_RAG                        Mixtral_RAG_Filt          \
stat                  average     max  median    min          average     max   
length                 193.57  312.00  198.00  14.00           184.04  294.00   
unique_tokens          121.07  172.00  122.00  12.00           114.08  170.00   
ngram_diversity          0.85    0.98    0.87   0.67             0.85    0.94   
compression_ratio        0.55    1.01    0.56   0.42             0.54    0.64   
n_sentences             23.62   34.00   24.00   3.00            23.44   33.00   
mean_sent_len            8.22   12.12    8.23   4.67             7.88   11.89   
msttr                    0.73    0.87    0.74   0.54             0.73    0.85   
mattr                    0.73    0.84    0.74   0.56             0.72    0.84   
mtld                    93.07  201.82   91.59  26.46            88.36  201.82   
hdd                      0.86    0.91    0.87   0.72             0.85    0.91   
vocd                   114.29  202.97  118.57  38.92           109.92  202.97   
ttr                      0.64    0.82    0.65   0.44             0.63    0.82   
perplexity               7.58   39.41    7.42   3.34             6.90   14.10   

model                             
stat               median    min  
length             181.50  76.00  
unique_tokens      108.00  63.00  
ngram_diversity      0.86   0.68  
compression_ratio    0.54   0.43  
n_sentences         24.00  11.00  
mean_sent_len        7.87   5.26  
msttr                0.74   0.55  
mattr                0.74   0.56  
mtld                88.07  26.46  
hdd                  0.86   0.72  
vocd               109.72  38.92  
ttr              

In [17]:
def presentation_table_to_latex(
    presentation_table,
    stats=("average", "min", "max", "median"),
    filename="final_table_latex0.tex",
    float_format="%.2f",
):
    """
    Generate LaTeX from a presentation table with:
        rows = metrics
        columns = MultiIndex: model -> stat

    You can choose which stat columns to include using `stats`.

    Example:
        presentation_table_to_latex(
            presentation_table2,
            stats=("average", "median")
        )
    """

    # Always enforce this stat order
    stat_order = ["average", "min", "max", "median"]

    selected_stats = [s for s in stat_order if s in stats]

    if not isinstance(presentation_table.columns, pd.MultiIndex):
        raise ValueError("presentation_table must have MultiIndex columns: model -> stat")

    models = presentation_table.columns.get_level_values(0).unique()

    ordered_cols = []

    for model in models:
        for stat in selected_stats:
            col = (model, stat)
            if col in presentation_table.columns:
                ordered_cols.append(col)

    latex_table = presentation_table.loc[:, ordered_cols].to_latex(
        index=True,
        escape=True,
        multicolumn=True,
        multicolumn_format="c",
        float_format=lambda x: float_format % x if pd.notna(x) else "",
    )

    with open(filename, "w", encoding="utf-8") as f:
        f.write(latex_table)

    return latex_table

In [18]:
latex = presentation_table_to_latex(
    presentation_table2,
    stats=("median"),
    filename="final_teachers_table_all_median.tex"
)

print(latex)

\begin{tabular}{lrrrrrrrrr}
\toprule
model & Gold & MedGemma & MedGemma\_Filt & MedGemma\_RAG & MedGemma\_RAG\_Filt & Mixtral & Mixtral\_Filt & Mixtral\_RAG & Mixtral\_RAG\_Filt \\
stat & median & median & median & median & median & median & median & median & median \\
\midrule
length & 266.00 & 224.00 & 226.00 & 154.00 & 155.50 & 219.00 & 214.00 & 198.00 & 181.50 \\
unique\_tokens & 160.00 & 130.00 & 130.00 & 102.00 & 103.00 & 109.00 & 109.00 & 122.00 & 108.00 \\
ngram\_diversity & 0.84 & 0.85 & 0.85 & 0.89 & 0.89 & 0.77 & 0.78 & 0.87 & 0.86 \\
compression\_ratio & 0.54 & 0.51 & 0.51 & 0.57 & 0.57 & 0.46 & 0.46 & 0.56 & 0.54 \\
n\_sentences & 36.00 & 28.00 & 29.00 & 23.00 & 23.00 & 23.00 & 23.00 & 24.00 & 24.00 \\
mean\_sent\_len & 7.30 & 7.74 & 7.77 & 6.86 & 6.85 & 9.48 & 9.40 & 8.23 & 7.87 \\
msttr & 0.74 & 0.72 & 0.72 & 0.75 & 0.75 & 0.67 & 0.68 & 0.74 & 0.74 \\
mattr & 0.73 & 0.71 & 0.71 & 0.75 & 0.75 & 0.67 & 0.67 & 0.74 & 0.74 \\
mtld & 97.17 & 84.46 & 84.46 & 104.11 & 103.59 & 

In [19]:
latex = presentation_table_to_latex(
    presentation_table2,
    stats=("average"),
    filename="final_teachers_table_all_avg.tex"
)

print(latex)

\begin{tabular}{lrrrrrrrrr}
\toprule
model & Gold & MedGemma & MedGemma\_Filt & MedGemma\_RAG & MedGemma\_RAG\_Filt & Mixtral & Mixtral\_Filt & Mixtral\_RAG & Mixtral\_RAG\_Filt \\
stat & average & average & average & average & average & average & average & average & average \\
\midrule
length & 298.18 & 221.62 & 223.01 & 155.50 & 155.92 & 217.40 & 215.56 & 193.57 & 184.04 \\
unique\_tokens & 164.13 & 130.06 & 130.38 & 105.48 & 105.73 & 109.23 & 109.12 & 121.07 & 114.08 \\
ngram\_diversity & 0.84 & 0.85 & 0.85 & 0.89 & 0.89 & 0.76 & 0.76 & 0.85 & 0.85 \\
compression\_ratio & 0.54 & 0.51 & 0.51 & 0.57 & 0.57 & 0.46 & 0.46 & 0.55 & 0.54 \\
n\_sentences & 40.73 & 28.48 & 28.54 & 22.49 & 22.56 & 23.31 & 23.18 & 23.62 & 23.44 \\
mean\_sent\_len & 7.47 & 7.78 & 7.81 & 6.85 & 6.85 & 9.42 & 9.41 & 8.22 & 7.88 \\
msttr & 0.74 & 0.75 & 0.74 & 0.75 & 0.75 & 0.67 & 0.68 & 0.73 & 0.73 \\
mattr & 0.73 & 0.71 & 0.71 & 0.75 & 0.74 & 0.67 & 0.67 & 0.73 & 0.72 \\
mtld & 101.19 & 86.89 & 86.92 & 109.97 &